# FitLog Pro Analysis

Portfolio analytics notebook for workout performance, nutrition, body composition, goals, retention, and behaviour. All charts are saved to `notebooks/charts/`.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import psycopg2
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", palette="deep")

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = Path("notebooks")
PROJECT_ROOT = NOTEBOOK_DIR.parent
ENV_PATH = PROJECT_ROOT / "backend" / ".env"
CHART_DIR = NOTEBOOK_DIR / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

def read_env(path: Path) -> dict[str, str]:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            values[key.strip()] = value.strip().strip('"').strip("'")
    return values

def psycopg2_kwargs(database_url: str) -> dict[str, object]:
    sync_url = database_url.replace("postgresql+asyncpg://", "postgresql://")
    parsed = urlparse(sync_url)
    return {
        "dbname": parsed.path.lstrip("/"),
        "user": parsed.username,
        "password": parsed.password,
        "host": parsed.hostname,
        "port": parsed.port or 5432,
    }

env = read_env(ENV_PATH)
conn = psycopg2.connect(**psycopg2_kwargs(env["DATABASE_URL"]))

def q(sql: str, params: tuple | None = None) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn, params=params)

def savefig(name: str) -> None:
    plt.tight_layout()
    plt.savefig(CHART_DIR / name, dpi=160, bbox_inches="tight")
    plt.show()


## Section 1 — Workout Performance

### Progressive overload: weekly volume and rolling 4-week trend

In [ ]:
weekly_volume = q("""
    SELECT
        DATE_TRUNC('week', ws.started_at)::date AS week_start,
        e.name AS exercise,
        SUM(COALESCE(es.weight_kg, 0) * es.reps) AS volume_kg
    FROM exercise_sets es
    JOIN session_exercises se ON se.id = es.session_exercise_id
    JOIN workout_sessions ws ON ws.id = se.session_id
    JOIN exercises e ON e.id = se.exercise_id
    GROUP BY 1, 2
    ORDER BY 1, 2
""")

top_exercises = weekly_volume.groupby("exercise")["volume_kg"].sum().nlargest(5).index
plot_df = weekly_volume[weekly_volume["exercise"].isin(top_exercises)].copy()
plot_df["rolling_4w_volume"] = plot_df.groupby("exercise")["volume_kg"].transform(lambda s: s.rolling(4, min_periods=1).mean())

plt.figure(figsize=(12, 6))
sns.lineplot(data=plot_df, x="week_start", y="volume_kg", hue="exercise", alpha=0.35, legend=False)
sns.lineplot(data=plot_df, x="week_start", y="rolling_4w_volume", hue="exercise", linewidth=2.5)
plt.title("Progressive Overload: Weekly Volume with 4-Week Rolling Trend")
plt.xlabel("Week")
plt.ylabel("Volume (kg x reps)")
plt.xticks(rotation=30)
savefig("01_progressive_overload.png")
plot_df.head()


### PR tracker: max weight per exercise over time

In [ ]:
pr_data = q("""
    SELECT
        ws.started_at::date AS workout_date,
        e.name AS exercise,
        MAX(es.weight_kg) AS max_weight_kg
    FROM exercise_sets es
    JOIN session_exercises se ON se.id = es.session_exercise_id
    JOIN workout_sessions ws ON ws.id = se.session_id
    JOIN exercises e ON e.id = se.exercise_id
    WHERE es.weight_kg IS NOT NULL
    GROUP BY 1, 2
    ORDER BY 1
""")

pr_plot = pr_data[pr_data["exercise"].isin(top_exercises)].copy()
pr_plot["historical_pr"] = pr_plot.groupby("exercise")["max_weight_kg"].cummax()
pr_events = pr_plot[pr_plot["max_weight_kg"].eq(pr_plot["historical_pr"])]

plt.figure(figsize=(12, 6))
sns.scatterplot(data=pr_events, x="workout_date", y="max_weight_kg", hue="exercise", s=55)
for _, row in pr_events.sort_values("workout_date").tail(10).iterrows():
    plt.annotate(f"{row['max_weight_kg']:.0f}kg", (row["workout_date"], row["max_weight_kg"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.title("PR Tracker: Personal Record Events by Exercise")
plt.xlabel("Date")
plt.ylabel("Max Weight (kg)")
plt.xticks(rotation=30)
savefig("02_pr_tracker.png")
pr_events.head()


### Volume heatmap: sessions per week × muscle group

In [ ]:
muscle_sessions = q("""
    SELECT DISTINCT
        DATE_TRUNC('week', ws.started_at)::date AS week_start,
        ec.muscle_group,
        ws.id AS session_id
    FROM workout_sessions ws
    JOIN session_exercises se ON se.session_id = ws.id
    JOIN exercises e ON e.id = se.exercise_id
    JOIN exercise_categories ec ON ec.id = e.category_id
""")
heatmap_df = muscle_sessions.pivot_table(index="muscle_group", columns="week_start", values="session_id", aggfunc="nunique", fill_value=0)

plt.figure(figsize=(14, 5))
sns.heatmap(heatmap_df, cmap="YlGnBu", linewidths=0.1)
plt.title("Workout Volume Heatmap: Sessions per Week by Muscle Group")
plt.xlabel("Week")
plt.ylabel("Muscle Group")
savefig("03_volume_heatmap.png")
heatmap_df.head()


### Rest vs performance: rest seconds and next-set reps

In [ ]:
rest_perf = q("""
    SELECT
        se.id AS session_exercise_id,
        es.set_number,
        es.rest_seconds,
        es.reps,
        LEAD(es.reps) OVER (PARTITION BY se.id ORDER BY es.set_number) AS next_set_reps
    FROM exercise_sets es
    JOIN session_exercises se ON se.id = es.session_exercise_id
    WHERE es.rest_seconds IS NOT NULL
""")
rest_perf = rest_perf.dropna(subset=["rest_seconds", "next_set_reps"])
rest_corr = rest_perf["rest_seconds"].corr(rest_perf["next_set_reps"])

plt.figure(figsize=(9, 6))
sns.regplot(data=rest_perf.sample(min(len(rest_perf), 3000), random_state=42), x="rest_seconds", y="next_set_reps", scatter_kws={"alpha": 0.25}, line_kws={"color": "crimson"})
plt.title(f"Rest vs Next-Set Reps (Pearson r = {rest_corr:.2f})")
plt.xlabel("Rest Seconds")
plt.ylabel("Next-Set Reps")
savefig("04_rest_vs_performance.png")
rest_corr


## Section 2 — Nutrition

### Daily macro split vs recommended ratios

In [ ]:
daily_macros = q("""
    SELECT
        nl.log_date,
        SUM(fi.protein_g * mi.quantity_g / 100.0) AS protein_g,
        SUM(fi.carbs_g * mi.quantity_g / 100.0) AS carbs_g,
        SUM(fi.fat_g * mi.quantity_g / 100.0) AS fat_g
    FROM nutrition_logs nl
    JOIN meal_items mi ON mi.log_id = nl.id
    JOIN food_items fi ON fi.id = mi.food_item_id
    GROUP BY 1
    ORDER BY 1
""")
daily_macros["protein_cal"] = daily_macros["protein_g"] * 4
daily_macros["carbs_cal"] = daily_macros["carbs_g"] * 4
daily_macros["fat_cal"] = daily_macros["fat_g"] * 9
macro_plot = daily_macros.tail(30).set_index("log_date")[["protein_cal", "carbs_cal", "fat_cal"]]

ax = macro_plot.plot(kind="bar", stacked=True, figsize=(14, 6), width=0.9)
ax.set_title("Daily Macro Split: Last 30 Logged Days")
ax.set_xlabel("Date")
ax.set_ylabel("Estimated Macro Calories")
ax.legend(["Protein", "Carbs", "Fat"])
plt.xticks(rotation=75)
savefig("05_daily_macro_split.png")

macro_ratios = daily_macros[["protein_cal", "carbs_cal", "fat_cal"]].sum()
macro_ratios = (macro_ratios / macro_ratios.sum()).rename({"protein_cal": "Protein", "carbs_cal": "Carbs", "fat_cal": "Fat"})
recommended_ratios = pd.Series({"Protein": 0.30, "Carbs": 0.45, "Fat": 0.25})
pd.DataFrame({"actual": macro_ratios, "recommended": recommended_ratios})


### Rolling 7-day calorie average vs goal

In [ ]:
daily_calories = q("""
    SELECT log_date, SUM(total_calories) AS calories
    FROM nutrition_logs
    GROUP BY 1
    ORDER BY 1
""")
calorie_goal = q("""
    SELECT AVG(target_value) AS calorie_goal
    FROM goals
    WHERE goal_type IN ('daily_calories', 'calories', 'nutrition')
""")["calorie_goal"].iloc[0]
if pd.isna(calorie_goal):
    calorie_goal = 2200
daily_calories["rolling_7d"] = daily_calories["calories"].rolling(7, min_periods=1).mean()

plt.figure(figsize=(12, 6))
sns.lineplot(data=daily_calories, x="log_date", y="calories", alpha=0.25, label="Daily calories")
sns.lineplot(data=daily_calories, x="log_date", y="rolling_7d", linewidth=2.5, label="7-day average")
plt.axhline(calorie_goal, color="crimson", linestyle="--", label=f"Goal: {calorie_goal:.0f} kcal")
plt.title("Rolling 7-Day Calories vs Goal")
plt.xlabel("Date")
plt.ylabel("Calories")
plt.xticks(rotation=30)
plt.legend()
savefig("06_rolling_calories_vs_goal.png")
daily_calories.tail()


### Meal type distribution by time of day

In [ ]:
meal_dist = q("""
    SELECT meal_type, COUNT(*) AS logs
    FROM nutrition_logs
    GROUP BY 1
""")
time_map = {"breakfast": "Morning", "lunch": "Afternoon", "dinner": "Evening", "snack": "Snack / Flexible"}
meal_dist["time_of_day"] = meal_dist["meal_type"].str.lower().map(time_map).fillna("Unspecified")
time_dist = meal_dist.groupby("time_of_day", as_index=False)["logs"].sum().sort_values("logs", ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=time_dist, x="time_of_day", y="logs")
plt.title("Meal Type Distribution by Time of Day")
plt.xlabel("Time of Day")
plt.ylabel("Nutrition Logs")
savefig("07_meal_time_distribution.png")
time_dist


### Weekly food item diversity count per user

In [ ]:
food_diversity = q("""
    SELECT
        nl.user_id,
        DATE_TRUNC('week', nl.log_date)::date AS week_start,
        COUNT(DISTINCT mi.food_item_id) AS unique_foods
    FROM nutrition_logs nl
    JOIN meal_items mi ON mi.log_id = nl.id
    GROUP BY 1, 2
    ORDER BY 2
""")
weekly_diversity = food_diversity.groupby("week_start", as_index=False)["unique_foods"].mean()

plt.figure(figsize=(12, 5))
sns.lineplot(data=weekly_diversity, x="week_start", y="unique_foods", marker="o")
plt.title("Weekly Food Item Diversity per User")
plt.xlabel("Week")
plt.ylabel("Average Unique Foods")
plt.xticks(rotation=30)
savefig("08_food_diversity.png")
food_diversity.head()


## Section 3 — Body Composition & Goals

### Weight trend with 7-day EWMA smoothing

In [ ]:
body_daily = q("""
    SELECT logged_date, AVG(weight_kg) AS avg_weight_kg, AVG(body_fat_pct) AS avg_body_fat_pct
    FROM body_metrics
    GROUP BY 1
    ORDER BY 1
""")
body_daily["weight_ewm_7d"] = body_daily["avg_weight_kg"].ewm(span=7, adjust=False).mean()

plt.figure(figsize=(12, 5))
sns.lineplot(data=body_daily, x="logged_date", y="avg_weight_kg", alpha=0.3, label="Daily average")
sns.lineplot(data=body_daily, x="logged_date", y="weight_ewm_7d", linewidth=2.5, label="7-day EWMA")
plt.title("Weight Trend with 7-Day EWMA Smoothing")
plt.xlabel("Date")
plt.ylabel("Weight (kg)")
plt.xticks(rotation=30)
savefig("09_weight_ewma.png")
body_daily.tail()


### Dual-axis: weight + body fat %

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
sns.lineplot(data=body_daily, x="logged_date", y="avg_weight_kg", ax=ax1, color="steelblue", label="Weight")
sns.lineplot(data=body_daily, x="logged_date", y="avg_body_fat_pct", ax=ax2, color="darkorange", label="Body Fat %")
ax1.set_title("Weight and Body Fat % Timeline")
ax1.set_xlabel("Date")
ax1.set_ylabel("Weight (kg)", color="steelblue")
ax2.set_ylabel("Body Fat %", color="darkorange")
ax1.tick_params(axis="x", rotation=30)
fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.9))
savefig("10_weight_bodyfat_dual_axis.png")


### Goal completion rate by goal type

In [ ]:
goal_completion = q("""
    SELECT
        goal_type,
        AVG(CASE WHEN status = 'completed' THEN 1.0 ELSE 0.0 END) AS completion_rate,
        COUNT(*) AS goals
    FROM goals
    GROUP BY 1
    ORDER BY completion_rate DESC
""")

plt.figure(figsize=(9, 5))
sns.barplot(data=goal_completion, y="goal_type", x="completion_rate", orient="h")
plt.title("Goal Completion Rate by Goal Type")
plt.xlabel("Completion Rate")
plt.ylabel("Goal Type")
plt.xlim(0, 1)
savefig("11_goal_completion_rate.png")
goal_completion


### Goal pace and on-track flag

In [ ]:
goal_pace = q("""
    WITH latest_progress AS (
        SELECT DISTINCT ON (goal_id)
            goal_id,
            current_value,
            recorded_date
        FROM goal_progress
        ORDER BY goal_id, recorded_date DESC
    )
    SELECT
        g.id,
        g.goal_type,
        g.target_value,
        g.target_date,
        g.status,
        u.created_at::date AS start_date,
        COALESCE(lp.current_value, 0) AS current_value,
        COALESCE(lp.recorded_date, CURRENT_DATE) AS recorded_date
    FROM goals g
    JOIN users u ON u.id = g.user_id
    LEFT JOIN latest_progress lp ON lp.goal_id = g.id
    WHERE g.target_date IS NOT NULL AND g.target_value > 0
""")
today = pd.Timestamp.today().normalize()
goal_pace["start_date"] = pd.to_datetime(goal_pace["start_date"])
goal_pace["target_date"] = pd.to_datetime(goal_pace["target_date"])
goal_pace["days_elapsed"] = (today - goal_pace["start_date"]).dt.days.clip(lower=1)
goal_pace["total_days"] = (goal_pace["target_date"] - goal_pace["start_date"]).dt.days.clip(lower=1)
goal_pace["pace_ratio"] = (goal_pace["current_value"] / goal_pace["target_value"]) / (goal_pace["days_elapsed"] / goal_pace["total_days"])
goal_pace["on_track"] = goal_pace["pace_ratio"] >= 1

pace_summary = goal_pace.groupby("goal_type", as_index=False)["on_track"].mean()
plt.figure(figsize=(9, 5))
sns.barplot(data=pace_summary, y="goal_type", x="on_track", orient="h")
plt.title("Goal Pace: Share of Goals On Track")
plt.xlabel("On-Track Rate")
plt.ylabel("Goal Type")
plt.xlim(0, 1)
savefig("12_goal_pace_on_track.png")
goal_pace[["goal_type", "current_value", "target_value", "pace_ratio", "on_track"]].head()


## Section 4 — Retention & Behaviour

### Workout frequency distribution

In [ ]:
frequency = q("""
    SELECT
        user_id,
        DATE_TRUNC('week', started_at)::date AS week_start,
        COUNT(*) AS sessions
    FROM workout_sessions
    GROUP BY 1, 2
""")

plt.figure(figsize=(9, 5))
sns.histplot(frequency["sessions"], bins=range(1, int(frequency["sessions"].max()) + 3), discrete=True)
plt.title("Workout Frequency Distribution: Sessions per User per Week")
plt.xlabel("Sessions per User per Week")
plt.ylabel("User-Weeks")
savefig("13_workout_frequency_distribution.png")
frequency.describe()


### Streak analysis: consecutive active days per user

In [ ]:
active_days = q("""
    SELECT DISTINCT user_id, started_at::date AS active_date
    FROM workout_sessions
    ORDER BY user_id, active_date
""")
active_days["active_date"] = pd.to_datetime(active_days["active_date"])
active_days["day_number"] = active_days.groupby("user_id").cumcount()
active_days["streak_group"] = active_days["active_date"] - pd.to_timedelta(active_days["day_number"], unit="D")
streaks = active_days.groupby(["user_id", "streak_group"]).size().reset_index(name="streak_days")
max_streaks = streaks.groupby("user_id", as_index=False)["streak_days"].max()

plt.figure(figsize=(9, 5))
sns.histplot(max_streaks["streak_days"], bins=20)
plt.title("Workout Streak Analysis: Longest Active-Day Streak per User")
plt.xlabel("Longest Streak (days)")
plt.ylabel("Users")
savefig("14_streak_analysis.png")
max_streaks.sort_values("streak_days", ascending=False).head()


### Cohort retention matrix

In [ ]:
cohort = q("""
    SELECT
        u.id AS user_id,
        DATE_TRUNC('month', u.created_at)::date AS signup_month,
        DATE_TRUNC('month', ws.started_at)::date AS activity_month
    FROM users u
    LEFT JOIN workout_sessions ws ON ws.user_id = u.id
""")
cohort = cohort.dropna(subset=["activity_month"]).copy()
cohort["signup_month"] = pd.to_datetime(cohort["signup_month"])
cohort["activity_month"] = pd.to_datetime(cohort["activity_month"])
cohort["month_index"] = ((cohort["activity_month"].dt.year - cohort["signup_month"].dt.year) * 12 + (cohort["activity_month"].dt.month - cohort["signup_month"].dt.month)).clip(lower=0)
cohort_counts = cohort.groupby(["signup_month", "month_index"])["user_id"].nunique().reset_index()
cohort_sizes = cohort_counts[cohort_counts["month_index"].eq(0)][["signup_month", "user_id"]].rename(columns={"user_id": "cohort_size"})
retention = cohort_counts.merge(cohort_sizes, on="signup_month", how="left")
retention["retention_rate"] = retention["user_id"] / retention["cohort_size"]
retention_matrix = retention.pivot(index="signup_month", columns="month_index", values="retention_rate").fillna(0)

plt.figure(figsize=(12, 5))
sns.heatmap(retention_matrix, annot=True, fmt=".0%", cmap="Blues", vmin=0, vmax=1)
plt.title("Workout Cohort Retention Matrix")
plt.xlabel("Months Since Signup")
plt.ylabel("Signup Month")
savefig("15_cohort_retention_matrix.png")
retention_matrix


### Churn prediction: logistic regression and ROC-AUC

In [ ]:
churn_features = q("""
    WITH workout_stats AS (
        SELECT
            u.id AS user_id,
            COUNT(ws.id)::float / GREATEST((CURRENT_DATE - MIN(ws.started_at::date)) / 7.0, 1) AS avg_sessions_per_week,
            CURRENT_DATE - MAX(ws.started_at::date) AS days_since_last_session
        FROM users u
        LEFT JOIN workout_sessions ws ON ws.user_id = u.id
        GROUP BY u.id
    ), goal_stats AS (
        SELECT
            user_id,
            AVG(CASE WHEN status = 'completed' THEN 1.0 ELSE 0.0 END) AS goal_completion_rate
        FROM goals
        GROUP BY user_id
    )
    SELECT
        ws.user_id,
        COALESCE(ws.avg_sessions_per_week, 0) AS avg_sessions_per_week,
        COALESCE(ws.days_since_last_session, 999) AS days_since_last_session,
        COALESCE(gs.goal_completion_rate, 0) AS goal_completion_rate
    FROM workout_stats ws
    LEFT JOIN goal_stats gs ON gs.user_id = ws.user_id
""")
churn_features["churned"] = churn_features["days_since_last_session"] > 30
X = churn_features[["avg_sessions_per_week", "days_since_last_session", "goal_completion_rate"]]
y = churn_features["churned"].astype(int)

if y.nunique() < 2 or len(churn_features) < 20:
    print("Not enough class balance for churn model. Need both churned and retained users.")
    auc = None
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, scores)
    fpr, tpr, _ = roc_curve(y_test, scores)

    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.title("Churn Prediction ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    savefig("16_churn_prediction_roc_auc.png")

auc


In [ ]:
conn.close()
